# 多工具 Agent · Anthropic 原生 Tool Use

**目标**：用结构化的 tool use 实现一个带 3 个工具的 Agent：

- `calculator(expr)`：表达式求值
- `now(tz)`：时区时间
- `unit_convert(value, from_unit, to_unit)`：单位换算

对比上一章「自然语言 ReAct」的脆弱解析，看看 schema-based tool use 优势何在。

In [ ]:
import os, sys, json, datetime
from zoneinfo import ZoneInfo
sys.path.append(os.path.abspath('../..'))

from anthropic import Anthropic
client = Anthropic()
MODEL = 'claude-sonnet-4-5'

## 1. 工具实现

In [ ]:
import math

def t_calculator(expr: str) -> str:
    try:
        return str(eval(expr, {'__builtins__': {}}, {'math': math}))
    except Exception as e:
        return f'Error: {e}'

def t_now(tz: str) -> str:
    try:
        return datetime.datetime.now(ZoneInfo(tz)).isoformat(timespec='seconds')
    except Exception as e:
        return f'Error: {e}'

_UNIT = {  # SI 基准换算（极简）
    'km': 1000.0, 'm': 1.0, 'cm': 0.01, 'mile': 1609.344, 'foot': 0.3048,
}
def t_unit_convert(value: float, from_unit: str, to_unit: str) -> str:
    try:
        return str(value * _UNIT[from_unit] / _UNIT[to_unit])
    except KeyError as e:
        return f'Unsupported unit: {e}'

DISPATCH = {'calculator': t_calculator, 'now': t_now, 'unit_convert': t_unit_convert}

## 2. 工具 schema（Anthropic 格式）

In [ ]:
TOOLS = [
    {
        'name': 'calculator',
        'description': '安全求值 Python 数学表达式，例如 calculator("3*(4+5)") 返回 27',
        'input_schema': {
            'type': 'object',
            'properties': {'expr': {'type': 'string'}},
            'required': ['expr'],
        },
    },
    {
        'name': 'now',
        'description': '返回指定 IANA 时区的当前 ISO 时间，例如 tz="Asia/Shanghai"',
        'input_schema': {
            'type': 'object',
            'properties': {'tz': {'type': 'string'}},
            'required': ['tz'],
        },
    },
    {
        'name': 'unit_convert',
        'description': '在 km, m, cm, mile, foot 之间换算长度',
        'input_schema': {
            'type': 'object',
            'properties': {
                'value': {'type': 'number'},
                'from_unit': {'type': 'string'},
                'to_unit': {'type': 'string'},
            },
            'required': ['value', 'from_unit', 'to_unit'],
        },
    },
]

## 3. Agent 主循环

In [ ]:
def run_agent(user_query: str, max_steps: int = 8, verbose: bool = True):
    messages = [{'role': 'user', 'content': user_query}]
    for step in range(max_steps):
        resp = client.messages.create(
            model=MODEL, max_tokens=512, tools=TOOLS, messages=messages,
        )
        if verbose:
            for b in resp.content:
                if b.type == 'text':
                    print(f'[step {step}] text: {b.text}')
                elif b.type == 'tool_use':
                    print(f'[step {step}] tool_use: {b.name}({b.input})')
        if resp.stop_reason != 'tool_use':
            return ''.join(b.text for b in resp.content if b.type == 'text')
        messages.append({'role': 'assistant', 'content': resp.content})
        results = []
        for b in resp.content:
            if b.type == 'tool_use':
                fn = DISPATCH.get(b.name)
                out = fn(**b.input) if fn else f'unknown tool {b.name}'
                if verbose:
                    print(f'[step {step}] obs: {out}')
                results.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': str(out)})
        messages.append({'role': 'user', 'content': results})
    return '[max_steps reached]'

print(run_agent('上海现在是几点？以小时为单位算一下离明天 0 点还有多久。'))

In [ ]:
print(run_agent('一辆车 65 mile/h 跑 2.5 小时，换算成 km 是多少？'))

## 4. 观察

- LLM 自动决定调几个、调哪个工具，且参数总是正确的 schema。
- 与上一章「自然语言 ReAct」相比，**不再需要正则解析**——这就是 Function Calling 的工程价值。
- 实际生产里：把 `DISPATCH` 替换为 MCP client，agent 即可调用任何 MCP server 的 tools。

## 进阶练习

1. 把 3 个工具搬到 `mcp_demo/` 写成一个 MCP server，让本 agent 通过 MCP client 调用。
2. 加一层 `parallel_tool_use=true`（Anthropic 支持并行 tool 调用），观察一次推理调多个工具的情况。
3. 故意让用户提一个无法靠工具解决的问题，看 agent 如何处理。